# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: Scoring (ranking-flavored).** Lane 4 (CTR / Engagement Opportunity Scoring) isn't
"will this page decline" (classification) or "what groups of pages exist" (clustering) — it's
"of the pages we already know rank somewhere, which ones are leaving clicks on the table, and
how many?" That's a decision about *ordering*, not a yes/no label, so it maps to scoring: assign
every eligible page a number, sort, work the queue top to bottom.

I'm calling it scoring rather than pure ranking because the number itself carries meaning
(estimated clicks/90d left on the table) and not just an ordinal position — an editor can look
at rank #1's score and rank #50's score and know the first is worth roughly 10x more attention,
which a bare ranking (1st, 2nd, 3rd...) wouldn't tell them.

## 2. Target or proxy

**Proxy, not an observed label — and that's an honest constraint, not a shortcut.** There is no
column in this data (or the warehouse) that says "this page's CTR gap is a real, fixable
problem." So the target is a *defined* proxy: the gap between a page's own CTR and the median
CTR of other pages in its **position tier** (`tier_median_ctr - ctr`), converted into an
estimated absolute number via `× impressions_90d` → "lost clicks/90d".

Two things keep this proxy honest instead of circular:

- **It's relative, not a flat threshold.** A flat `ctr < 0.5%` rule mostly just re-discovers
  position tier (CTR craters naturally past `top_3`) — comparing each page only to peers in the
  *same* tier isolates the part of the gap that isn't explained by ranking alone.
- **I check it against a signal it was never built from.** `engagement_rate` (also tier-median-
  compared) plays no role in computing the score, so a queue item where engagement is *also*
  weak is a page where two independent metrics agree something's off — not proof, but real
  corroboration rather than the rule confirming itself.

## 3. Success metric

**Precision@K against the engagement-corroboration proxy**, evaluated at a few K values (20,
50, 100, 200) and compared to the base rate so "good" isn't a number I picked after seeing the
result. Precision@K here means: *of the top K pages by lost-clicks score, what fraction also
show below-tier-median engagement_rate?* Since there's no ground-truth "this was worth fixing"
label, I can't report a real precision/recall against reality — only a lift over the base rate,
read as "meaningfully enriched," never as "accurate." I name this metric before building the
queue below, precisely so I can't quietly redefine "good" once I see the numbers.

## 4. The unit of analysis, as a real dataframe

**One row = one content item (a single page for one client), scored against the eligible pool
of visible, high-volume pages** (`impressions_90d >= 500`, `avg_position` reported and `<= 20`
— the same pool the ML-02 discovery numbers used). Loading the slice below and adding the
tier-relative gap columns so the target proxy is visible as actual data, not just prose.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Lane 4's eligible pool: visible pages with enough traffic to trust a CTR comparison.
# avg_position == 0 means "no position data" (per the data dictionary), not rank zero -- exclude it.
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()

print(f"Full starter dataset: {len(df):,} rows (one row = one content item)")
print(f"Lane 4 eligible pool: {len(pool):,} rows ({len(pool)/len(df):.1%} of the dataset)")
print()
print("Unit of analysis check -- one row per content_id within the pool:")
print(f"  unique content_id in pool: {pool.content_id.nunique():,} / {len(pool):,} rows")

id_cols = ["content_id", "client_id", "position_tier", "avg_position"]
metric_cols = ["impressions_90d", "clicks_90d", "ctr", "engagement_rate"]
pool[id_cols + metric_cols].head(8)

Full starter dataset: 30,000 rows (one row = one content item)
Lane 4 eligible pool: 12,023 rows (40.1% of the dataset)

Unit of analysis check -- one row per content_id within the pool:
  unique content_id in pool: 12,023 / 12,023 rows


,content_id,client_id,position_tier,avg_position,impressions_90d,clicks_90d,ctr,engagement_rate
0,content_304f48230142,client_f369cb89fc,striking,10.6,3803,29,0.76,5.88
3,content_331d6c4de07b,client_19581e27de,page_1,6.2,11751,58,0.49,1.28
5,content_d4084a4bc775,client_f369cb89fc,page_1,8.5,3970,1,0.03,0.00
9,content_c27558df2b0c,client_19581e27de,page_1,4.9,1240,2,0.16,0.00
10,content_d8ee6cc6d642,client_19581e27de,top_3,2.2,20919,324,1.55,6.75
12,content_42fb2cad9ecf,client_6208ef0f77,page_1,5.6,7228,127,1.76,3.43
16,content_78bd1d4a1d4d,client_6208ef0f77,page_1,8.9,13848,21,0.15,0.21
17,content_761a44afda12,client_19581e27de,page_1,7.3,9449,7,0.07,11.86


**Sketching the target column.** Below, the tier-relative gap and the lost-clicks-90d proxy
target, plus the engagement corroboration check used for the success metric -- built directly on
the dataframe above, not a separate mocked-up example.

In [2]:
# Compare each page only within its own position tier (a top_3 page is not judged
# against a striking page -- position alone moves CTR far too much for a flat comparison).
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")
pool["tier_median_engagement"] = pool.groupby("position_tier")["engagement_rate"].transform("median")

pool["ctr_gap"] = (pool["tier_median_ctr"] - pool["ctr"]).clip(lower=0)
pool["target_lost_clicks_90d"] = (pool["ctr_gap"] / 100) * pool["impressions_90d"]
pool["engagement_confirms"] = (pool.engagement_rate > 0) & (pool.engagement_rate < pool.tier_median_engagement)

queue = pool[pool.ctr < pool.tier_median_ctr].sort_values(
    "target_lost_clicks_90d", ascending=False
).reset_index(drop=True)

print(f"Opportunity queue (below own tier's median CTR): {len(queue):,} / {len(pool):,} pages "
      f"({len(queue)/len(pool):.1%})")
print()
show_cols = ["content_id", "position_tier", "impressions_90d", "ctr", "tier_median_ctr",
             "ctr_gap", "target_lost_clicks_90d", "engagement_confirms"]
print("Highest-scoring rows (the target column, made real):")
display_df = queue[show_cols].head(10)
display_df

# Precision@K vs base rate -- the success metric named in section 3, computed here, not
# invented after looking at what looks good.
base_rate = queue["engagement_confirms"].mean()
print(f"\nBase rate (engagement-confirmed weak page, full queue): {base_rate:.3f}")
for k in (20, 50, 100, 200):
    p = queue["engagement_confirms"].head(k).mean()
    print(f"precision@{k:<4} = {p:.3f}   (lift x{p / base_rate:.1f} over base rate)")

Opportunity queue (below own tier's median CTR): 5,888 / 12,023 pages (49.0%)

Highest-scoring rows (the target column, made real):

Base rate (engagement-confirmed weak page, full queue): 0.005
precision@20   = 0.050   (lift x9.8 over base rate)
precision@50   = 0.100   (lift x19.6 over base rate)
precision@100  = 0.080   (lift x15.7 over base rate)
precision@200  = 0.045   (lift x8.8 over base rate)


## 5. Why ML beats a fixed rule here

**Honestly? For this version, it doesn't yet -- and that's the point of framing before modeling.**
The tier-relative gap rule above already beats a *flat* rule (a flat `ctr < 0.5%` threshold
mostly just rediscovers position tier, since CTR craters with position regardless of anything
else -- see the ML-02 discovery numbers). But comparing a page only to its own tier's median is
still an if-statement a person could write and defend today. It earns a place as the **baseline**,
not as evidence ML is needed.

Where a fixed rule starts to strain, and a learned model could earn its place:

- **Interacting signals a single threshold can't hold at once.** `content_type`,
  `word_count`, `has_ai_traffic`, and tier all shift what "normal" CTR looks like
  *simultaneously* -- a rule can add one more `if` per interaction, but that doesn't scale past
  two or three signals before it becomes an unreadable pile of thresholds.
- **The rule is currently blind to a shifting baseline.** "Median CTR per tier" is static per
  snapshot; a model could learn how much of a gap is *typical drift* vs. a real outlier, using
  more of the 44 columns than the four the rule touches.
- **The weak-pick review from ML-07 already shows the rule's failure mode**: it's dominated by
  raw impression volume and content-type concentration (see `w04_baseline_score.ipynb`) -- a
  model with proper regularization could down-weight volume instead of letting it dominate the
  score outright.

So the honest claim for this notebook: the rule is the right *first* answer, and it also
generates the evaluation harness (the tier-median comparison, the engagement-corroboration
check) that any later model has to beat before it's allowed to replace the rule.